In [1]:
# this notebook will serve as a space for developing functions for x-fuse
# it will be used to test out code snippets and ideas before they are added to the main codebase

In [2]:
# Plan out the steps for the x-fuse pipeline below

# STEP 1: GET DINO FEATURES

# STEP 2: UPSAMPLE DINO FEATURES INTEGRATING WITH XRD DATA

# STEP 3: SEGMENT USING UPSAMPLED DINO FEATURES

In [3]:
# set the cache directories

import os
torch_dir = '/dls/science/groups/imaging/infuse/XRDCT_Fusion/torch_cache/'
os.environ["HUGGINGFACE_HUB_CACHE"] = torch_dir
os.environ["TORCH_HOME"] = torch_dir
os.environ["HF_HUB_CACHE"] = torch_dir
os.environ["HF_HOME"] = torch_dir

In [4]:
# imports

import torch
torch.cuda.empty_cache()


from hr_dv2.utils import *
from hr_dv2 import tr
from hr_dv2 import HighResDV2
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from typing import Tuple, List, Any, TypedDict, TypeAlias

from hr_dv2.segment import get_dv2_features

from PIL import Image

import imageio.v3 as iio

In [5]:
!nvidia-smi

Mon Apr 20 15:32:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100S-PCIE-32GB          On  |   00000000:06:00.0 Off |                    0 |
| N/A   35C    P0             54W /  250W |       4MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 1. Get DINO Features

In [6]:
IMG_SIZE = 630
PATCH_SIZE = 14 #DINOv2
PATCH_H, PATCH_W = IMG_SIZE // PATCH_SIZE, IMG_SIZE // PATCH_SIZE

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [7]:
# intialise the HighResDV2 model
dino_model_name = "dinov2_vitl14"
net = HighResDV2(dino_model_name, 4, pca_dim=-1, dtype=torch.float16)
net.to(DEVICE)

FEAT_DIM = net.get_model_params(dino_model_name)[0]
print(FEAT_DIM)

Using cache found in /dls/science/groups/imaging/infuse/XRDCT_Fusion/torch_cache/hub/facebookresearch_dinov2_main
/dls/science/groups/imaging/infuse/XRDCT_Fusion/torch_cache/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/dls/science/groups/imaging/infuse/XRDCT_Fusion/torch_cache/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/dls/science/groups/imaging/infuse/XRDCT_Fusion/torch_cache/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


1024


In [8]:
# load the image

os.chdir("/dls/science/groups/imaging/infuse/XRDCT_Fusion/X-FUSE/notebooks")

img_path = "../x_fuse/datasets/Zn13X/high_4.tiff"

_img = iio.imread(img_path)
h,w = _img.shape[0], _img.shape[1]

img_tr= tr.closest_crop(h, w, PATCH_SIZE)

img_tr = tr.get_input_transform(IMG_SIZE, IMG_SIZE)

img_tensor, img = tr.load_image(img_path, img_tr)
img_arr = np.array(img)

img_tensor =img_tensor.to(torch.float16).to(DEVICE)




In [9]:
shift_dists = [i for i in range(1,3)]

fwd_shift, inv_shift = tr.get_shift_transforms(shift_dists, 'Moore')
fwd_flip, inv_flip = tr.get_flip_transforms()
fwd, inv = tr.combine_transforms(fwd_shift, fwd_flip, inv_shift, inv_flip)

net.set_transforms(fwd, inv)

In [10]:
print(net.n_heads)
print(net.dinov2.num_heads)

6
16


In [11]:
!nvidia-smi

Mon Apr 20 15:33:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100S-PCIE-32GB          On  |   00000000:06:00.0 Off |                    0 |
| N/A   34C    P0             36W /  250W |     934MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
attn_choice = 'none'
feats_attn_tensor = net.forward_sequential(img_tensor, attn_choice=attn_choice)
print(feats_attn_tensor.shape)
# if attn_choice != "none":
#     feats, attn = feats_attn_tensor[0, :-6, :, :], feats_attn_tensor[0, -6:, :, :]
# else:
feats = feats_attn_tensor[0]

OutOfMemoryError: CUDA out of memory. Tried to allocate 17.21 GiB. GPU 0 has a total capacity of 31.73 GiB of which 12.36 GiB is free. Including non-PyTorch memory, this process has 19.37 GiB memory in use. Of the allocated memory 18.96 GiB is allocated by PyTorch, and 35.85 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
feats_np = tr.to_numpy(feats)
feats_flat = tr.flatten(feats_np, feats_np.shape[1], feats_np.shape[2], feats_np.shape[0])

In [ ]:
pcaed = do_single_pca(feats_flat, 3, n_samples=5000)

feats_np.shape

In [ ]:
rescaled = rescale_pca(pcaed)

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2)
fig.set_size_inches(18, 8)

axs[0].imshow(_img, cmap='gray')
print(_img.shape)
axs[1].imshow(rescaled.reshape(630, 630, 3), interpolation='nearest')
print(rescaled.reshape(630, 630, 3).shape)
# axs[2].imshow(np.sum(tr.to_numpy(attn), axis=0) , interpolation='nearest')

titles = ["Original", "Feature PCA", "sum [CLS] attn"]
for i, ax in enumerate(axs):
    ax.set_axis_off()
    ax.set_title(titles[i], fontsize=20)
plt.tight_layout()

In [ ]:
# different mechanism for integrating XRD

FUNCTION direct_correlation_gating(features, xrd_map):
    // features: (27, 27, 384)
    // xrd_map: (27, 27, 1)
    
    FOR each channel ch in 0..384:
        correlation[ch] = compute_correlation(
            features[:, :, ch],  // all spatial locations for channel ch
            xrd_map              // XRD intensity map
        )
    END FOR
    
    gates = clamp(correlation, 0, 1)  // (384,)
    
    FOR each spatial location (i, j):
        features[i, j, :] = features[i, j, :] * gates
    END FOR
    
    RETURN features


In [ ]:
CLASS LearnedChannelGating(feat_dim=384):
    
    INIT:
        mlp = neural_network([
            Linear(1 → 128),
            ReLU(),
            Linear(128 → feat_dim),
            Sigmoid()  // output [0, 1]
        ])
    
    FORWARD(features, xrd_map):
        // features: (27, 27, 384)
        // xrd_map: (27, 27, 1)
                FOR each spatial location (i, j):
            xrd_intensity = xrd_map[i, j]  // scalar
            
            gates = mlp(xrd_intensity)  // (384,)
            
            features[i, j, :] = features[i, j, :] * gates
        END FOR
        
        RETURN features
    
    TRAIN:
        FOR each image in dataset:
            gated_features = forward(features, xrd_map)
            upsampled = upsample(gated_features)
            
            loss = correlation_distance(
                upsampled_pca,
                xrd_map
            )
            
            backprop(loss)

In [ ]:
CLASS XRDCrossAttention(feat_dim=384, num_heads=8):
    
    INIT:
        xrd_to_query = Linear(1 → feat_dim)
        attention_layer = MultiHeadAttention(
            embed_dim=feat_dim,
            num_heads=num_heads
        )
    
    FORWARD(features, xrd_map):
        // features: (729, 384)  [flattened 27×27]
        // xrd_map: (729, 1)      [flattened]
        
        // Project XRD to query space
        queries = xrd_to_query(xrd_map)  // (729, 384)
        
        // Attention: "Where does XRD signal attend in feature space?"
        attended_features, attention_weights = attention(
            query = queries,        // What XRD is looking for
            key = features,         // Which features to attend to
            value = features        // What to retrieve
        )  // (729, 384)
        
        RETURN attended_features